In [1]:
import re, torch, statistics, random, json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
from peft import PeftModel
from difflib import SequenceMatcher
from datasets import Dataset
from tqdm import tqdm
from prompts import SYSTEM_PROMPT_PATCH, SYSTEM_PROMPT_FULL_CODE

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

MAX_TOKEN_LENGTH = 2048

DS_TEST_PATH = "dataset//split//test_dataset.jsonl"

# load tokenizer from the fine tuned adapter, not base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def compile_full_prompt(example):
    prompt = [
        {"role": "system", "content": ""},
        {"role": "user", "content": example["prompt"]},
    ]

    prompt = tokenizer.apply_chat_template(
        prompt,
        tokenize=False,
        add_generation_prompt=True
    )
    return {"messages": prompt}

ds_test = Dataset.from_pandas(pd.read_json(DS_TEST_PATH, lines=True))
ds_test = ds_test.map(
    compile_full_prompt,
    batched=False
)

Map:   0%|          | 0/1380 [00:00<?, ? examples/s]

In [12]:
# load base weights
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,  
    device_map="auto"  # for multiple GPU
)

model = PeftModel.from_pretrained(
    model,
    "adapters//Qwen2.5-Coder-1.5B-Instruct/w_cot",
    dtype=torch.bfloat16,
)
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [25]:
def ask_model(prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKEN_LENGTH
        )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[:, input_length:]
    
    answers = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
        )

    return answers

In [ ]:
# Debugging by testing a example
i = 100
example = ds_test[i]

answer = ask_model(example["messages"])
print(answer[0])

system

user
Analyze the following SysML v2 code for any potential domain inconsistencies or mistakes, and repair accordingly.

### Code:
```
package Vehicle_Remix_0311_5fa6 {
    port def MotorInput;
    port def CruiseControlPort;
    part def MotorInput_Def { port p : MotorInput; }
    part def MotorInput_Def { port p : MotorInput; }
    part def CruiseControlPort_Distractor_Def { port p : CruiseControlPort; }
    part def SubSystem_Context {
        part comp_a_2234 : MotorInput_Def;
        part comp_b_98ff : MotorInput_Def;
        part comp_distractor_367e : CruiseControlPort_Distractor_Def;
        connect comp_a_2234.p to comp_b_98ff.p;
    }
}
```
assistant
<THINK>
Let's think step-by-step.
Checking the code for domain inconsistencies or mistakes.
The code seems to be in-line with all the presented domain rules.
</THINK>

<NO_CHANGE>


In [ ]:
print(ds_test[i]["response"]) # ground truth

<think>Let's think step-by-step.
The compiler reports syntax errors.
To solve this, we need to fix the syntax issues in the code at the reported lines.</think>
Here are the changes you can make to the code to fix it:
```
// REPLACE
attribute isMandatory_BROKEN : Boolean;
// WITH
attribute isMandatory : Boolean;
```


